# PromptGuard + Qwen v3 탐지 평가 (Colab GPU)

PromptGuard(promptguard-claude-demoV0)에 `ml-qwen`만 켠 구성 하나를 **전체 데이터 8종**으로 평가한다. 서버(ARM CPU)에서는 약 20시간 걸리고 타임아웃 위험이 있어 이 구성만 Colab GPU로 옮겼다. 평가 경로는 서버 `run_config.py`와 같은 `dashboard/pipeline/runner.evaluate`이고, 결과 `summary.json`은 서버 `report.py`로 같은 표를 만들 수 있다.

**결과를 읽을 때:** GPU에서 돌렸으므로 **지연과 타임아웃 차단은 서버 결과와 직접 비교할 수 없다.** 타임아웃(25초)은 그대로지만, 서버에서라면 막혔을 입력이 GPU에서는 통과할 수 있다. 차단된 항목은 전체가 `over`로 채점되므로 탐지 점수도 이 조건의 영향을 받는다.

**시스템 사본에서 바꾸는 설정:** `src/detector/detector.json`의 `ml-qwen.enabled`(true)와 `ml-qwen.options.repository`(Colab 경로)뿐이다. `ml-v9`는 꺼 두고, 나머지 설정은 그대로 두며 `PG_ML_TIMEOUT_SECONDS`도 쓰지 않는다.

**준비물:** GPU 런타임, Google Drive의 `MyDrive/promptguard/pg-data-full-07cf988.tar.gz`(`README.md`의 "데이터 올리기"). 옛 `pg-data-full.tar.gz`는 해시 검사에서 멈춘다.

**순서:** ① 설정 → ② 레포 → ③ 패키지 → ④ gitleaks → ⑤ 데이터 → ⑥ 시스템 설정·사전 점검 → ⑦ 작은 실행·시간 어림 → ⑧ 전체 실행 → ⑨ 진행 확인 → ⑩ 결과. 처음에는 "모두 실행"으로 ⑦까지 본 뒤, ①의 `RUN_FULL`을 `True`로 바꿔 ①과 ⑧을 실행한다.

**끊겼을 때:**
- 브라우저만 끊겼거나 커널만 다시 시작됐으면(런타임 유지) ①과 ⑨만 실행한다. 평가는 별도 프로세스라 계속 돈다. 이때 ②~⑥을 실행하면 멈추도록 해 두었다.
- 런타임이 초기화됐으면 ①부터 다시 실행한다. 끝난 샤드는 Drive 체크포인트에서 이어 가고, 돌던 샤드 하나만 다시 계산한다.

**비밀값:** 압축 파일, `resume/`, `results/*/examples/`에는 비밀값 형식의 문자열이 있다. 공유 폴더에 두지 않는다. 이 노트북은 개수와 지표만 출력한다.

## ① 설정

In [ ]:
# ── 바꿔도 되는 값 ─────────────────────────────────────────────
RUN_FULL = False  # ⑦의 시간 어림을 본 뒤 True로 바꾼다. 실수로 긴 실행이 시작되지 않게 기본은 False다
TIMING_SAMPLES_PER_DATASET = 40  # ⑦에서 데이터셋마다 시간을 재는 항목 수

# ── 고정값: 서버 결과와 비교하려면 바꾸지 않는다. 아래 단계가 실행 중에 검증하고, 다르면 멈춘다 ─────
SYSTEM_COMMIT = "cca44c4fdb9c1a059d3491a67bb00e7b52aadf2d"  # promptguard-claude-demoV0
MODEL_COMMIT = "84099234bc7bd96b680b0a77225ace57553c1f9b"   # promptguard-model (models/qwen-span-v3)
SCORER_COMMIT = "9bd0d37437123a743957a38ac4c2ca910576946a"  # promptguard-data, eval/mod-parity-cca44c4: 현재 Mod 규칙, gitleaks 캐시 키
BASE_MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
BASE_REVISION = "ea3f2471cf1b1f0db85067f1ef93848e38e88c25"
THRESHOLD = 0.9824353001249998
ADAPTER_WEIGHTS = "models/qwen-span-v3/adapter_model.safetensors"
ADAPTER_WEIGHTS_SHA256 = "913a3f589d9221fa3a79ea65e87e4550283ca44149cdb5a48d88cf6d7d85dc0a"
DATA_SHA256 = "1d762bbf187a259a0f13505201796bd43b57239a9f288ddcd447e4a78dca6944"  # 라벨 정제 후 현재 데이터 8종
EXPECTED_BENCHMARK = "673148fe347d5d198a9276be571c47421cf6004818e75bd3fc56a17aecc75f8f"  # 서버 "ML 없음" 결과의 지문. 같으면 데이터·채점 코드·gitleaks 버전이 같다
EXPECTED_CALLS = 20111  # 전체 데이터의 Qwen 호출 수
GITLEAKS_VERSION = "8.30.1"
GITLEAKS_SHA256 = "551f6fc83ea457d62a0d98237cbad105af8d557003051f41f3e7ca7b3f2470eb"  # linux x64, 릴리스 checksums
PACKAGES = ["credsweeper==1.19.0", "transformers==4.57.6", "peft>=0.13,<1", "PyYAML>=6,<7"]  # torch는 Colab 기본 설치본을 쓴다
EXPECTED_OPTIONS = {"candidate_policy": "union", "bundle": "models/qwen-span-v3", "timeout_seconds": 25,
                    "max_text_chars": 12000, "too_long": "skip", "gate": "cue", "device": "auto"}
SMOKE_STEMS = ["sessions", "sessions_from_example", "sessions_2"]

# ── 경로 ───────────────────────────────────────────────────────
ROOT = "/content"
DRIVE_ROOT = f"{ROOT}/drive"
DATA_ARCHIVE = f"{DRIVE_ROOT}/MyDrive/promptguard/pg-data-full-07cf988.tar.gz"
DRIVE_DIR = f"{DRIVE_ROOT}/MyDrive/promptguard/qwen-v3-cca44c4"  # 체크포인트와 결과. 공유 폴더에 두지 않는다
LOCAL_DIR = f"{ROOT}/pg-qwen"  # 런타임 로컬(gitleaks 캐시, 임시 파일). Drive에 올리지 않는다
SYSTEM_ROOT, MODEL_ROOT, SCORER_ROOT = f"{ROOT}/promptguard-claude-demoV0", f"{ROOT}/promptguard-model", f"{ROOT}/promptguard-data"
HF_HOME = f"{LOCAL_DIR}/hf-cache"
DRIVER, CONFIG = f"{LOCAL_DIR}/driver.py", f"{LOCAL_DIR}/config.json"
GITHUB = "https://github.com/capstone-privai"

import fcntl, hashlib, json, os, shlex, subprocess, sys, time
from pathlib import Path

def sh(command, env=None):
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True, env=env)

def is_running(name):
    """드라이버가 쥐고 있는 잠금으로 판단한다. 프로세스가 죽으면 잠금도 풀린다."""
    path = Path(LOCAL_DIR, f"driver-{name}.lock")
    if not path.exists():
        return False
    with open(path, "w") as handle:
        try:
            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
            return False
        except OSError:
            return True

def assert_idle():
    if is_running("smoke") or is_running("qwen"):
        raise RuntimeError("평가가 돌고 있어 환경을 바꾸지 않는다. ⑨로 가서 진행을 확인한다.")

def run_driver(*args):
    done = subprocess.run([sys.executable, DRIVER, CONFIG, *args], capture_output=True, text=True)
    if done.returncode != 0:
        print(done.stdout)
        print("\n".join(done.stderr.strip().splitlines()[-5:]))
        raise RuntimeError(f"드라이버 {args[0]} 실패 (exit {done.returncode})")
    return done.stdout

def show_status(name):
    status = json.loads(Path(DRIVE_DIR, f"status_{name}.json").read_text(encoding="utf-8"))
    done, total = status.get("done"), status.get("total")
    progress = f"{done:,}/{total:,} ({done / total:.1%})" if done is not None and total else "-"
    print(time.strftime("%H:%M:%S"), name, status["state"], progress, "| 샤드", status["shards_done"], "| blocked", status["blocked"],
          "| failed_open", status["failed_open"], "| adapter 실패", status["adapter_failures"], flush=True)
    return status

def launch(name):
    """별도 세션의 프로세스로 띄운다. 셀을 중단하거나 브라우저가 끊겨도 평가는 계속된다."""
    if is_running(name):
        print(name, "이미 실행 중")
        return
    log = open(Path(DRIVE_DIR, f"driver_{name}.log"), "a")
    subprocess.Popen([sys.executable, DRIVER, CONFIG, "run", "--name", name], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    log.close()
    print(name, "시작")
    time.sleep(10)

def driver_events(name, count=3):
    path = Path(DRIVE_DIR, f"driver_{name}.log")
    lines = [line for line in path.read_text(encoding="utf-8", errors="replace").splitlines() if line.startswith('{"event"')] if path.is_file() else []
    return lines[-count:]

def watch(name, every=300):
    """이 셀을 중단해도 평가는 멈추지 않는다. 다시 실행하면 이어서 본다."""
    last = 0
    while is_running(name):
        if time.time() - last >= every:
            last = time.time()
            if Path(DRIVE_DIR, f"status_{name}.json").is_file():
                show_status(name)
        time.sleep(15)
    status = show_status(name) if Path(DRIVE_DIR, f"status_{name}.json").is_file() else {"state": "not started"}
    if status["state"] != "complete":
        print("\n".join(driver_events(name)))
        raise RuntimeError(f"{name} 실행이 끝나지 않았다({status['state']}). 위 원인을 확인한 뒤 launch로 다시 시작하면 끝난 샤드부터 이어 간다.")

print("평가 실행 중" if is_running("smoke") or is_running("qwen") else "실행 중인 평가 없음", "| RUN_FULL =", RUN_FULL)

## ② 레포 받기

세 레포를 고정 커밋으로 받는다. `clone -b`로는 커밋을 지정할 수 없어 `fetch` 후 `checkout`한다. 가중치는 Qwen v3 파일만 Git LFS로 받고, 내용 해시를 확인한다.

채점기 커밋(`9bd0d37`)은 `main`에 없다. GitHub에 push된 뒤에 받을 수 있고, 없으면 이 셀이 안내하고 멈춘다.

In [ ]:
assert_idle()
if subprocess.run("git lfs version", shell=True, capture_output=True).returncode != 0:
    sh("apt-get -qq install -y git-lfs")
sh("git lfs install")
env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}  # 다른 모델의 가중치까지 받지 않는다

def fetch_commit(name, path, commit):
    if subprocess.run(["git", "-C", path, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() == commit:
        print(name, "이미 받음", commit[:12])
        return
    sh(f"rm -rf {shlex.quote(path)} && git init -q {shlex.quote(path)} && git -C {shlex.quote(path)} remote add origin {GITHUB}/{name}.git")
    fetched = subprocess.run(["git", "-C", path, "fetch", "-q", "--depth", "1", "origin", commit], env=env, capture_output=True, text=True)
    if fetched.returncode != 0:
        raise RuntimeError(f"{name}에서 커밋 {commit}을 받지 못했다. GitHub에 push됐는지 확인한다: {fetched.stderr.strip()}")
    sh(f"git -C {shlex.quote(path)} checkout -q --detach {commit}", env=env)

for name, path, commit in (("promptguard-claude-demoV0", SYSTEM_ROOT, SYSTEM_COMMIT), ("promptguard-model", MODEL_ROOT, MODEL_COMMIT),
                           ("promptguard-data", SCORER_ROOT, SCORER_COMMIT)):
    fetch_commit(name, path, commit)
    head = subprocess.run(["git", "-C", path, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
    assert head == commit, f"{name} HEAD {head} != {commit}"
    print(name, head)

sh(f"git -C {MODEL_ROOT} lfs pull --include={ADAPTER_WEIGHTS} --exclude=''")
digest = hashlib.sha256(Path(MODEL_ROOT, ADAPTER_WEIGHTS).read_bytes()).hexdigest()
assert digest == ADAPTER_WEIGHTS_SHA256, f"가중치 해시가 다르다: {digest}"
print("Qwen v3 가중치 확인:", Path(MODEL_ROOT, ADAPTER_WEIGHTS).stat().st_size, "bytes")

## ③ 패키지와 GPU

설치한 패키지는 평가 프로세스가 새로 읽으므로, 경고가 나와도 세션을 다시 시작하지 않아도 된다. GPU가 없으면 여기서 멈춘다. CPU로 돌면 서버보다 느리고 타임아웃 차단이 늘어나기 때문이다.

In [ ]:
assert_idle()
sh(f"{shlex.quote(sys.executable)} -m pip -q install " + " ".join(shlex.quote(package) for package in PACKAGES))
CHECK = """
import importlib.metadata as md, json, torch
print(json.dumps({"versions": {name: md.version(name) for name in ("credsweeper", "transformers", "peft", "torch", "numpy")},
                  "cuda": torch.cuda.is_available(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))
"""
checked = json.loads(subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1])
print(json.dumps(checked, ensure_ascii=False))
assert checked["versions"]["credsweeper"] == "1.19.0", checked["versions"]
assert checked["versions"]["transformers"] == "4.57.6", checked["versions"]
assert checked["cuda"], "GPU가 없다. 런타임 유형을 GPU로 바꾼 뒤 ①부터 다시 실행한다."

## ④ gitleaks

서버와 같은 8.30.1을 linux x64 릴리스에서 받아 해시를 확인한다. 서버의 `gitleaks-cache.sqlite`는 비밀값이 들어 있고 ARM 바이너리 해시로 키가 묶여 여기서 적중하지 않으므로 올리지 않는다. 캐시는 이 런타임 안에서 새로 만든다.

In [ ]:
assert_idle()
archive = f"/tmp/gitleaks_{GITLEAKS_VERSION}_linux_x64.tar.gz"
sh(f"curl -fsSL -o {archive} https://github.com/gitleaks/gitleaks/releases/download/v{GITLEAKS_VERSION}/gitleaks_{GITLEAKS_VERSION}_linux_x64.tar.gz")
digest = hashlib.sha256(Path(archive).read_bytes()).hexdigest()
assert digest == GITLEAKS_SHA256, f"gitleaks 해시가 다르다: {digest}"
sh(f"tar -xzf {archive} -C /usr/local/bin gitleaks")
version = subprocess.run(["gitleaks", "version"], capture_output=True, text=True, check=True).stdout.strip()
assert version == GITLEAKS_VERSION, f"gitleaks {version}"
print("gitleaks", version)

## ⑤ 데이터

Drive의 압축 파일 해시를 먼저 확인한다. 다르면 멈춘다. 옛 `pg-data-full.tar.gz`를 쓰면 라벨이 정제되기 전 데이터로 평가되기 때문이다.

In [ ]:
assert_idle()
from google.colab import drive
drive.mount(DRIVE_ROOT)

assert Path(DATA_ARCHIVE).is_file(), f"{DATA_ARCHIVE}가 없다. 서버의 runs/colab/pg-data-full-07cf988.tar.gz를 Drive에 올린다."
digest = hashlib.sha256(Path(DATA_ARCHIVE).read_bytes()).hexdigest()
assert digest == DATA_SHA256, f"압축 파일 해시가 다르다: {digest}. 옛 파일이 아닌지 확인한다."
target = Path(SCORER_ROOT, "metric-detection")
sh(f"rm -rf {target}/data_test {target}/data_answer && tar -xzf {shlex.quote(DATA_ARCHIVE)} -C {target}")
stems = sorted(path.stem for path in (target / "data_test").glob("sessions*.jsonl"))
assert len(stems) == 8, stems
print("데이터", len(stems), "종:", ", ".join(stems))
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
Path(LOCAL_DIR).mkdir(parents=True, exist_ok=True)

## ⑥ 시스템 설정과 사전 점검

`detector.json`은 커밋의 내용에서 매번 같은 바이트로 다시 만든다. 그래서 `system_fingerprint`가 재시작해도 같고 체크포인트를 다시 쓸 수 있다. 베이스 모델을 고정 revision으로 미리 받은 뒤 평가는 오프라인으로 읽는다. 샤드마다 모델을 새로 읽는데, 네트워크 오류로 읽기에 실패하면 Qwen이 빠진 채 점수가 나올 수 있기 때문이다.

점검 항목은 커밋, 어댑터 `ml-qwen`·`gitleaks` 동작, `ml-v9` 꺼짐, 임계값, 타임아웃, 베이스 revision, GPU 사용이다. 하나라도 어긋나면 멈춘다.

드라이버 코드는 아래 셀에 그대로 들어 있고 런타임 로컬에만 저장된다.

In [ ]:
DRIVER_SOURCE = r"""# Qwen v3 평가 드라이버. 노트북이 만들어 별도 프로세스로 돌린다. 런타임 연결이 끊겨도 체크포인트에서 이어 간다.
# 평가는 서버 run_config.py와 같은 runner.evaluate 경로로 한다. 데이터 내용과 비밀값은 출력하지 않는다.
import argparse, fcntl, hashlib, json, os, random, re, shutil, statistics, subprocess, sys, time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

CONFIG_PATH = Path(sys.argv[1])
cfg = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
SYSTEM, SCORER, MODEL = Path(cfg["system_root"]), Path(cfg["scorer_root"]), Path(cfg["model_root"])
DRIVE, LOCAL = Path(cfg["drive_dir"]), Path(cfg["local_dir"])
DATA = SCORER / "metric-detection" / "data_test"
DETECTOR_JSON = SYSTEM / "src" / "detector" / "detector.json"
ML, OTHER_ML = "ml-qwen", "ml-v9"
ADAPTER_FAILED = re.compile(r"adapter '([^']+)' failed \((\w+)\)")
SHARD_NAME = re.compile(r"^(.+)_\d{6}-\d{6}$")


def stop(message):
    print(json.dumps({"event": "stop", "reason": message}, ensure_ascii=False), flush=True)
    sys.exit(3)


def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def read_json(path, default=None):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return default


def git(repo, *args):
    return subprocess.run(["git", "-C", str(repo), *args], check=True, capture_output=True, text=True).stdout


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def verify_repos():
    problems = []
    for repo, commit in ((SYSTEM, cfg["system_commit"]), (SCORER, cfg["scorer_commit"]), (MODEL, cfg["model_commit"])):
        head = git(repo, "rev-parse", "HEAD").strip()
        if head != commit:
            problems.append(f"{repo.name} HEAD {head} != {commit}")
    for repo, allowed in ((SYSTEM, {"src/detector/detector.json"}), (SCORER, set()), (MODEL, set())):
        changed = {line[3:] for line in git(repo, "status", "--porcelain", "--untracked-files=no").splitlines()}
        if changed - allowed:
            problems.append(f"{repo.name} 수정된 파일 {sorted(changed - allowed)}")
    if problems:
        stop("커밋 또는 작업 트리가 고정값과 다르다: " + "; ".join(problems))


def flatten(value, prefix=""):
    if isinstance(value, dict):
        return {key: leaf for name, item in value.items() for key, leaf in flatten(item, f"{prefix}{name}.").items()}
    if isinstance(value, list):
        return {key: leaf for index, item in enumerate(value)
                for key, leaf in flatten(item, f"{prefix}{item['name'] if isinstance(item, dict) and 'name' in item else index}.").items()}
    return {prefix[:-1]: value}


def cmd_configure(_args):
    # HEAD의 detector.json에서 항상 새로 만든다. 같은 입력이면 같은 바이트라 system_fingerprint가 재시작해도 같다.
    original = json.loads(git(SYSTEM, "show", "HEAD:src/detector/detector.json"))
    config = json.loads(json.dumps(original))
    adapters = {adapter["name"]: adapter for adapter in config["adapters"]}
    adapters[ML]["enabled"] = True
    adapters[ML]["options"]["repository"] = str(MODEL)
    adapters[OTHER_ML]["enabled"] = False
    before, after = flatten(original), flatten(config)
    changes = {key: [before.get(key), after.get(key)] for key in sorted(set(before) | set(after))
               if before.get(key) != after.get(key)}
    allowed = {f"adapters.{ML}.enabled", f"adapters.{ML}.options.repository", f"adapters.{OTHER_ML}.enabled"}
    if set(changes) - allowed:
        stop(f"허용하지 않은 설정 변경: {sorted(set(changes) - allowed)}")
    options = adapters[ML]["options"]
    fixed = {"candidate_policy": config.get("candidate_policy"), "bundle": options.get("bundle"),
             "timeout_seconds": options.get("timeout_seconds"), "max_text_chars": options.get("max_text_chars"),
             "too_long": options.get("too_long"), "gate": options.get("gate"), "device": options.get("device")}
    if fixed != cfg["expected_options"]:
        stop(f"커밋의 설정이 예상과 다르다: {fixed} != {cfg['expected_options']}")
    DETECTOR_JSON.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    verify_repos()
    print(json.dumps({"event": "configured", "changes": changes, "kept": fixed,
                      "detector_json_sha256": sha256_file(DETECTOR_JSON)}, ensure_ascii=False, indent=1), flush=True)


def setup_scorer():
    sys.path.insert(0, str(SCORER / "dashboard"))
    from pipeline import runner
    return runner


def load_detector():
    setup_scorer()
    from evaluation.adapters.system_root import load_worker
    return load_worker(SYSTEM).load_detector()


def describe_ml(adapter):
    inner = adapter.model.detector
    return {"threshold": adapter.model.threshold, "timeout": adapter.timeout, "model_id": adapter.model_id,
            "revision": adapter.model.meta["revision"], "dtype": adapter.model.meta["dtype"],
            "device": str(inner.device), "parameter_dtype": str(next(inner.model.parameters()).dtype)}


def cmd_probe(_args):
    started = time.perf_counter()
    detector = load_detector()
    info = detector.describe()
    out = {"adapters": [adapter["name"] for adapter in info["adapters"]], "skipped": info["skipped"],
           "candidate_policy": info["candidate_policy"], "load_seconds": round(time.perf_counter() - started, 1)}
    for adapter in detector.adapters:
        if adapter.name == ML:
            out["ml"] = describe_ml(adapter)
    print(json.dumps(out, ensure_ascii=False), flush=True)


def run_probe():
    done = subprocess.run([sys.executable, __file__, str(CONFIG_PATH), "probe"], capture_output=True, text=True)
    if done.returncode != 0:
        stop(f"탐지기 로드 실패 (exit {done.returncode}): {done.stderr.strip().splitlines()[-1] if done.stderr.strip() else ''}")
    return json.loads(done.stdout.strip().splitlines()[-1])


def check_probe(info):
    problems = []
    if ML not in info["adapters"]:
        problems.append(f"{ML} 없음")
    if "gitleaks" not in info["adapters"]:
        problems.append("gitleaks 없음")
    if OTHER_ML in info["adapters"]:
        problems.append(f"{OTHER_ML}가 켜져 있음")
    if info["skipped"]:
        problems.append(f"빠진 어댑터 {info['skipped']}")
    ml = info.get("ml") or {}
    if abs(float(ml.get("threshold", -1)) - cfg["threshold"]) > 1e-12:
        problems.append(f"임계값 {ml.get('threshold')} != {cfg['threshold']}")
    if ml.get("timeout") != cfg["expected_options"]["timeout_seconds"]:
        problems.append(f"타임아웃 {ml.get('timeout')}")
    if ml.get("revision") != cfg["base_revision"]:
        problems.append(f"베이스 revision {ml.get('revision')}")
    if cfg["require_gpu"] and not str(ml.get("device", "")).startswith("cuda"):
        problems.append(f"GPU가 아닌 장치 {ml.get('device')}")
    if problems:
        stop("사전 점검 실패: " + "; ".join(problems))


def environment():
    versions = {}
    for package in ("torch", "transformers", "peft", "credsweeper", "numpy", "tokenizers"):
        try:
            versions[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            versions[package] = None
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True) if shutil.which("nvidia-smi") else None
    gitleaks = subprocess.run(["gitleaks", "version"], capture_output=True, text=True).stdout.strip()
    return {"gpu": smi.stdout.strip() if smi and smi.returncode == 0 else None,
            "python": sys.version.split()[0], "gitleaks": gitleaks, **versions}


def record_environment(name):
    # 세션이 끊겨 이어 돌리면 GPU 종류나 패키지가 바뀔 수 있다. 조건이 바뀌면 경고를 남긴다.
    path = DRIVE / f"environment_{name}.json"
    history = read_json(path, [])
    current = environment()
    if history and any(item["environment"] != current for item in history):
        print(json.dumps({"event": "warning", "reason": "이전 세션과 실행 환경이 다르다", "before": history[-1]["environment"],
                          "now": current}, ensure_ascii=False), flush=True)
    history.append({"at": now(), "environment": current})
    write_json(path, history)
    return history


def iter_texts(path, channels):
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                for item in json.loads(line)["items"]:
                    if item["channel"] in channels:
                        yield item["text"], channels[item["channel"]]


def call_counts():
    # 데이터셋별 스캔 항목 수와 Qwen 호출 수. 어댑터의 판단(공백 제외, 단서 단어, 길이 한도)과 같은 조건으로 센다.
    path = DRIVE / "calls.json"
    saved = read_json(path)
    if saved and saved.get("data_sha256") == cfg["data_sha256"] and saved.get("counts"):
        return saved["counts"]
    runner = setup_scorer()
    from evaluation.adapters.promptguard_adapter import CHANNEL_MAP
    from evaluation.adapters.system_root import load_worker
    load_worker(SYSTEM)
    from pgdetector.adapters.span_models import CUE
    limit = cfg["expected_options"]["max_text_chars"]
    counts = {}
    for dataset in runner.discover(DATA):
        row = {"items": 0, "calls": 0, "too_long": 0, "chars": 0}
        for text, _source in iter_texts(dataset.sessions, CHANNEL_MAP):
            row["items"] += 1
            if text.strip() and CUE.search(text):
                if len(text) > limit:
                    row["too_long"] += 1
                else:
                    row["calls"] += 1
                    row["chars"] += len(text)
        counts[dataset.stem] = row
    write_json(path, {"data_sha256": cfg["data_sha256"], "counts": counts})
    return counts


def cmd_timing(args):
    # 데이터셋마다 항목을 무작위로 뽑아 탐지기 전체를 재고, 그 평균으로 전체 시간을 어림한다(비 ML 탐지기 포함).
    counts = call_counts()
    total_calls = sum(row["calls"] for row in counts.values())
    if total_calls != cfg["expected_calls"]:
        print(json.dumps({"event": "warning", "reason": f"Qwen 호출 수 {total_calls} != 예상 {cfg['expected_calls']}"},
                         ensure_ascii=False), flush=True)
    started = time.perf_counter()
    detector = load_detector()
    load_seconds = time.perf_counter() - started
    adapter = next(item for item in detector.adapters if item.name == ML)
    check_probe({"adapters": [item.name for item in detector.adapters], "skipped": detector.describe()["skipped"],
                 "ml": describe_ml(adapter)})
    from evaluation.adapters.promptguard_adapter import CHANNEL_MAP
    from pgdetector.adapters.span_models import CUE
    limit = cfg["expected_options"]["max_text_chars"]
    detector.scan("DB_PASSWORD=warmup123\n", "warmup")  # 첫 호출의 준비 시간을 재지 않는다
    rng = random.Random(cfg["timing_seed"])
    rows, all_calls = {}, []
    for stem in sorted(counts):
        texts = list(iter_texts(DATA / f"{stem}.jsonl", CHANNEL_MAP))
        chosen = rng.sample(texts, min(args.per_dataset, len(texts)))
        item_seconds, call_seconds, failures = [], [], 0
        for text, source in chosen:
            began = time.perf_counter()
            result = detector.scan(text, source)
            elapsed = time.perf_counter() - began
            item_seconds.append(elapsed)
            if text.strip() and CUE.search(text) and len(text) <= limit:
                call_seconds.append(elapsed)
            failures += 0 if result.get("ok") else 1
        all_calls += call_seconds
        rows[stem] = {"sampled": len(chosen), "failures": failures, "item_mean_s": round(statistics.fmean(item_seconds), 3),
                      "sampled_calls": len(call_seconds),
                      "call_mean_s": round(statistics.fmean(call_seconds), 3) if call_seconds else None,
                      "call_max_s": round(max(call_seconds), 3) if call_seconds else None,
                      "estimate_s": round(statistics.fmean(item_seconds) * counts[stem]["items"], 1),
                      "items": counts[stem]["items"], "calls": counts[stem]["calls"]}
    load_total = load_seconds * cfg["estimated_shards"]
    estimate = sum(row["estimate_s"] for row in rows.values()) + load_total
    payload = {"at": now(), "rows": rows, "load_seconds": round(load_seconds, 1), "total_calls": total_calls,
               "call_mean_s": round(statistics.fmean(all_calls), 3) if all_calls else None,
               "call_median_s": round(statistics.median(all_calls), 3) if all_calls else None,
               "call_max_s": round(max(all_calls), 3) if all_calls else None,
               "estimate_hours": round(estimate / 3600, 2)}
    write_json(DRIVE / "timing.json", payload)
    print(json.dumps(payload, ensure_ascii=False), flush=True)


class Tracker:
    # 끝난 샤드를 바로 검사하고 진행 상황을 Drive에 남긴다. ml-qwen은 required가 아니라, 어느 샤드 프로세스에서
    # 모델 로드가 실패하면 Qwen 없이 점수가 나온다. 그런 샤드는 합치지 않고 멈춘다.

    def __init__(self, name, checkpoint_dir, out_dir, error_type):
        self.name, self.checkpoint_dir, self.out_dir, self.error_type = name, checkpoint_dir, out_dir, error_type
        self.path = DRIVE / f"status_{name}.json"
        self.failures_path = DRIVE / f"adapter_failures_{name}.json"
        self.failures = read_json(self.failures_path, {})
        self.seen, self.shards, self.bad = {}, {}, {}
        self.started = time.time()
        self.first_done = None

    @staticmethod
    def problem(payload):
        detector = payload["system"]["detector"]
        names = [adapter["name"] for adapter in detector["adapters"]]
        if ML not in names or "gitleaks" not in names or OTHER_ML in names or detector["skipped"]:
            return f"adapters={names} skipped={detector['skipped']}"
        return None

    def sweep(self):
        for path in sorted(self.checkpoint_dir.glob("*.json")) if self.checkpoint_dir.is_dir() else []:
            stamp = path.stat().st_mtime_ns
            if self.seen.get(path.name) == stamp:
                continue
            saved = read_json(path)
            if saved is None:
                continue  # 아직 동기화 중이거나 깨진 파일. runner도 같은 파일을 쓰지 않고 다시 계산한다
            self.seen[path.name] = stamp
            payload = saved["payload"]
            self.shards[path.stem] = {"blocked": payload["system"].get("blocked_items", 0),
                                      "failed_open": payload["system"].get("failed_open_items", 0),
                                      "items": sum(row[2] for row in payload["items"]), "seconds": payload["seconds"]}
            reason = self.problem(payload)
            if reason:
                self.bad[path.name] = reason

    def quarantine(self):
        # 문제 샤드는 이어 하기에 재사용되지 않게 옮긴다(다음 실행이 다시 계산한다).
        self.sweep()
        for filename, reason in self.bad.items():
            target = DRIVE / "quarantine" / self.name / f"{int(time.time())}-{filename}"
            target.parent.mkdir(parents=True, exist_ok=True)
            source = self.checkpoint_dir / filename
            if source.is_file():
                shutil.move(str(source), str(target))
            self.shards.pop(Path(filename).stem, None)
            print(json.dumps({"event": "quarantine", "shard": filename, "reason": reason}, ensure_ascii=False), flush=True)
        moved = dict(self.bad)
        self.bad.clear()
        return moved

    def read_logs(self):
        # 어댑터 실패는 이름과 예외 형식만 샤드 로그에 남는다. 로그는 임시 폴더와 함께 지워지므로 개수만 옮겨 둔다.
        changed = False
        for shard in self.out_dir.parent.glob(f"{self.out_dir.name}.work-*/shards/*.json"):
            log = shard.parent.parent / "logs" / f"{shard.stem}.log"
            if not log.is_file():
                continue  # 체크포인트에서 복원한 샤드
            found = {}
            for match in ADAPTER_FAILED.finditer(log.read_text(encoding="utf-8", errors="replace")):
                key = f"{match.group(1)}:{match.group(2)}"
                found[key] = found.get(key, 0) + 1
            if self.failures.get(shard.stem) != found:
                self.failures[shard.stem] = found
                changed = True
        if changed:
            write_json(self.failures_path, self.failures)

    def status(self, state, done=None, total=None, error=None):
        failures = {}
        for shard in self.failures.values():
            for key, value in shard.items():
                failures[key] = failures.get(key, 0) + value
        write_json(self.path, {"name": self.name, "state": state, "updated_at": now(), "done": done, "total": total,
                               "shards_done": len(self.shards), "elapsed_s": round(time.time() - self.started),
                               "blocked": sum(s["blocked"] for s in self.shards.values()),
                               "failed_open": sum(s["failed_open"] for s in self.shards.values()),
                               "adapter_failures": failures, "error": error})

    def progress(self, done, total):
        self.read_logs()
        self.sweep()
        self.status("running", done, total)
        print(json.dumps({"event": "progress", "name": self.name, "done": done, "total": total, "shards": len(self.shards),
                          "seconds": round(time.time() - self.started)}), flush=True)
        if self.bad:
            raise self.error_type(f"Qwen이나 gitleaks가 빠진 샤드: {sorted(self.bad)}")


def write_notes(name, summary, history):
    environments = {json.dumps(item["environment"], sort_keys=True): item["environment"] for item in history}
    lines = [
        f"# PromptGuard + Qwen v3 Colab 실행 조건 ({name})", "",
        "서버 CPU가 아니라 Colab GPU에서 돌린 결과다. **지연과 타임아웃 차단은 서버 결과와 직접 비교할 수 없다.** "
        "타임아웃(25초)은 그대로지만 GPU가 빨라 서버에서라면 막혔을 입력이 통과할 수 있다. 차단은 해당 항목 전체를 "
        "`over`로 채점하므로 탐지 점수도 이 조건의 영향을 받는다.", "",
        "## 시스템 사본에서 바꾼 설정 (`src/detector/detector.json`)", "",
        f"- `ml-qwen.enabled`: false → true", f"- `ml-qwen.options.repository`: `../../../promptguard-model` → `{MODEL}`",
        "- `ml-v9`는 꺼 둔 그대로다. `candidate_policy`, `timeout_seconds`, `max_text_chars`, `too_long`은 바꾸지 않았고 "
        "`PG_ML_TIMEOUT_SECONDS`도 쓰지 않았다.", "",
        "## 고정값", "",
        f"- 시스템 `{cfg['system_commit']}`", f"- 모델 `{cfg['model_commit']}` (`models/qwen-span-v3`), 베이스 revision `{cfg['base_revision']}`, 임계값 {cfg['threshold']}",
        f"- 채점기 `{cfg['scorer_commit']}`", f"- 데이터 압축 파일 sha256 `{cfg['data_sha256']}`",
        f"- 벤치마크 지문 `{summary['benchmark']['fingerprint']}`", f"- 시스템 지문 `{summary['system']['fingerprint']}`", "",
        "## 실행 환경", "", "실행을 이어 간 세션의 환경을 모두 적었다. 둘 이상이면 샤드마다 조건이 다를 수 있다.", "",
        *[f"- `{json.dumps(item, ensure_ascii=False, sort_keys=True)}`" for item in environments.values()], "",
        "gitleaks 캐시는 이 런타임의 로컬 디스크에만 뒀다(서버 캐시 미사용). 베이스 모델은 미리 받아 두고 오프라인 모드로 읽었다.", "",
    ]
    return "\n".join(lines)


def publish(name, local, history):
    target = DRIVE / "results" / name
    staging = target.with_name(target.name + ".copying")
    shutil.rmtree(staging, ignore_errors=True)
    shutil.copytree(local, staging)
    summary = json.loads((staging / "summary.json").read_text(encoding="utf-8"))
    (staging / "RUN_NOTES.md").write_text(write_notes(name, summary, history), encoding="utf-8")
    shutil.rmtree(target, ignore_errors=True)
    os.replace(staging, target)
    return target


def cmd_run(args):
    name = args.name
    LOCAL.mkdir(parents=True, exist_ok=True)
    lock = open(LOCAL / f"driver-{name}.lock", "w")
    try:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except OSError:
        stop(f"{name} 실행이 이미 돌고 있다")
    runner = setup_scorer()
    verify_repos()
    info = run_probe()
    check_probe(info)
    print(json.dumps({"event": "preflight", "name": name, **info}, ensure_ascii=False), flush=True)
    history = record_environment(name)
    datasets = runner.discover(DATA)
    if len(datasets) != cfg["dataset_count"]:
        stop(f"데이터셋 {len(datasets)}종, 예상 {cfg['dataset_count']}종")
    bench_all = runner.benchmark(datasets)
    if bench_all["fingerprint"] != cfg["expected_benchmark"]:
        stop(f"벤치마크 지문 {bench_all['fingerprint']}가 서버 결과 {cfg['expected_benchmark']}와 다르다(데이터, 채점 코드, gitleaks 버전 중 하나가 다르다)")
    if name == "smoke":
        datasets = [dataset for dataset in datasets if dataset.stem in set(cfg["smoke_stems"])]
    original_plan = runner.plan
    runner.plan = lambda planned, weights, _jobs: original_plan(planned, weights, cfg["shards_for"])
    bench = runner.benchmark(datasets)
    config_hash = sha256_file(DETECTOR_JSON)
    fingerprint = f"{cfg['system_commit'][:12]}:qwen:{config_hash[:12]}"
    namespace = hashlib.sha256(f'{fingerprint}|{bench["fingerprint"]}'.encode()).hexdigest()
    identity = {"system_fingerprint": fingerprint, "benchmark_fingerprint": bench["fingerprint"], "namespace": namespace}
    identity_path = DRIVE / f"identity_{name}.json"
    saved_identity = read_json(identity_path)
    if saved_identity and saved_identity != identity:
        stop(f"이전 실행과 지문이 달라 체크포인트를 쓸 수 없다: {saved_identity} != {identity}")
    write_json(identity_path, identity)

    out_dir = LOCAL / "results" / name
    for stale in out_dir.parent.glob(f"{name}.work-*"):  # 강제 종료로 남은 임시 폴더. 잠금을 쥐고 있어 다른 실행의 것이 아니다
        shutil.rmtree(stale, ignore_errors=True)
    resume_dir = DRIVE / "resume" / name
    tracker = Tracker(name, resume_dir / namespace, out_dir, runner.EvaluationError)
    tracker.quarantine()
    tracker.status("starting")
    from pipeline.gitleaks_cache import GitleaksCache
    cache = LOCAL / "gitleaks-cache.sqlite"
    GitleaksCache(cache).close()
    commit = {"sha": cfg["system_commit"], "config": "qwen", "detector_json_sha256": config_hash,
              "model_commit": cfg["model_commit"], "scorer_commit": cfg["scorer_commit"], "data_sha256": cfg["data_sha256"],
              "run_environment": "colab-gpu" if cfg["require_gpu"] else "test"}
    started = time.time()
    try:
        headline = runner.evaluate(
            SYSTEM, datasets, out_dir, python=sys.executable, jobs=1, cache_dir=LOCAL / "validated", commit=commit,
            system_fingerprint=fingerprint, bench=bench, progress=tracker.progress, resume_dir=resume_dir,
            gitleaks_cache=cache)
    except BaseException as exc:
        tracker.quarantine()
        tracker.status("failed", error=str(exc)[:500])
        print(json.dumps({"event": "failed", "name": name, "error": str(exc)[:500]}, ensure_ascii=False), flush=True)
        raise SystemExit(1) from None
    tracker.read_logs()
    tracker.sweep()
    target = publish(name, out_dir, history)
    tracker.status("complete")
    print(json.dumps({"event": "complete", "name": name, "seconds": round(time.time() - started), "saved": str(target),
                      "precision": headline["precision"], "recall": headline["recall"]}, ensure_ascii=False), flush=True)


KEYS = ("exact", "over", "partial", "missed", "tp", "wide", "fp")


def check_summary(summary):
    # report.py의 load와 같은 검사다. 이게 통과하면 summary.json을 report.py로 읽을 수 있다.
    names = [dataset["key"] for dataset in summary["datasets"]]
    counts = {name: dict.fromkeys(KEYS, 0) for name in names}
    for row in summary["cells"]:
        for key, value in zip(KEYS, row[4:]):
            counts[names[row[0]]][key] += value
    totals = {key: sum(c[key] for c in counts.values()) for key in KEYS}
    expected = summary["totals"]["counts"]
    if (totals["exact"], totals["tp"], totals["wide"], totals["fp"]) != (expected["exact"], expected["edits_tp"], expected["edits_over"], expected["edits_fp"]):
        stop("summary.json의 칸 합계와 전체 합계가 다르다")


def cmd_report(args):
    name = args.name
    target = DRIVE / "results" / name
    summary = read_json(target / "summary.json")
    if summary is None:
        stop(f"{target}/summary.json이 없다. {name} 실행이 끝나지 않았다")
    check_summary(summary)
    calls = call_counts()
    failures = read_json(DRIVE / f"adapter_failures_{name}.json", {})
    per_dataset, known = {}, {}
    for shard, found in failures.items():
        stem = SHARD_NAME.match(shard).group(1)
        known[stem] = known.get(stem, 0) + 1
        for key, value in found.items():
            per_dataset.setdefault(stem, {}).setdefault(key, 0)
            per_dataset[stem][key] += value
    namespace = read_json(DRIVE / f"identity_{name}.json")["namespace"]
    shard_files = {}
    for path in (DRIVE / "resume" / name / namespace).glob("*.json"):
        stem = SHARD_NAME.match(path.stem).group(1)
        shard_files[stem] = shard_files.get(stem, 0) + 1
    print("| 데이터 | 세션 | 항목 | Qwen 호출(계산) | 정답 | 마스킹 | Precision | Recall | blocked | failed_open | adapter 실패 | 초 |")
    print("|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|---:|")
    prf = summary["totals"]["datasets"]
    missing = 0
    for dataset in summary["datasets"]:
        stem = dataset["key"]
        found = per_dataset.get(stem, {})
        missing += max(0, shard_files.get(stem, 0) - known.get(stem, 0))
        text = ", ".join(f"{key} {value}" for key, value in sorted(found.items())) or "0"
        precision, recall = prf[stem]["precision"], prf[stem]["recall"]
        print(f"| {stem} | {dataset['sessions']:,} | {dataset['items']:,} | {calls.get(stem, {}).get('calls', 0):,} | {dataset['gold']:,} | {dataset['edits']:,} "
              f"| {'—' if precision is None else format(precision, '.3f')} | {'—' if recall is None else format(recall, '.3f')} "
              f"| {dataset['blocked']} | {dataset['failed_open']} | {text} | {dataset['seconds']:,.0f} |")
    print()
    print(f"blocked 합계 {sum(d['blocked'] for d in summary['datasets'])}, failed_open 합계 {sum(d['failed_open'] for d in summary['datasets'])}, "
          f"실행 시간 {summary['wall_seconds'] / 3600:.2f}시간(이번 세션 기준)")
    if missing:
        print(f"주의: 샤드 {missing}개는 어댑터 실패 수를 기록하기 전에 세션이 끊겼다. adapter 실패 수가 그만큼 빠졌을 수 있다(blocked는 정확하다).")
    print(f"시스템 지문 {summary['system']['fingerprint']}, 벤치마크 지문 {summary['benchmark']['fingerprint']}")
    print(f"저장 위치: {target}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("config")
    commands = parser.add_subparsers(dest="command", required=True)
    commands.add_parser("configure")
    commands.add_parser("probe")
    timing = commands.add_parser("timing")
    timing.add_argument("--per-dataset", type=int, default=40)
    for command in ("run", "report"):
        sub = commands.add_parser(command)
        sub.add_argument("--name", required=True, choices=("smoke", "qwen"))
    args = parser.parse_args()
    leftover = [key for key in os.environ if key.startswith("PG_")]
    if leftover:
        stop(f"PG_ 환경 변수가 설정돼 있다. 지우고 다시 실행한다: {leftover}")
    os.environ["HF_HOME"] = cfg["hf_home"]
    if args.command != "configure" and cfg["offline"]:
        os.environ["HF_HUB_OFFLINE"] = "1"  # 샤드마다 모델을 다시 읽는다. 네트워크 오류로 Qwen이 빠지지 않게 받아 둔 사본만 쓴다
    {"configure": cmd_configure, "probe": cmd_probe, "timing": cmd_timing, "run": cmd_run, "report": cmd_report}[args.command](args)


if __name__ == "__main__":
    main()
"""

In [ ]:
assert_idle()
Path(LOCAL_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVER).write_text(DRIVER_SOURCE, encoding="utf-8")
Path(CONFIG).write_text(json.dumps({
    "system_root": SYSTEM_ROOT, "scorer_root": SCORER_ROOT, "model_root": MODEL_ROOT, "drive_dir": DRIVE_DIR, "local_dir": LOCAL_DIR,
    "hf_home": HF_HOME, "system_commit": SYSTEM_COMMIT, "scorer_commit": SCORER_COMMIT, "model_commit": MODEL_COMMIT,
    "base_revision": BASE_REVISION, "threshold": THRESHOLD, "data_sha256": DATA_SHA256, "expected_options": EXPECTED_OPTIONS,
    "require_gpu": True, "offline": True, "shards_for": 8, "smoke_stems": SMOKE_STEMS, "dataset_count": 8,
    "expected_benchmark": EXPECTED_BENCHMARK, "expected_calls": EXPECTED_CALLS, "estimated_shards": 30, "timing_seed": 20261106,
}, indent=1), encoding="utf-8")

print(run_driver("configure"))
DOWNLOAD = f"from huggingface_hub import snapshot_download; print(snapshot_download({BASE_MODEL!r}, revision={BASE_REVISION!r}))"
subprocess.run([sys.executable, "-c", DOWNLOAD], check=True, env={**os.environ, "HF_HOME": HF_HOME})
probe = json.loads(run_driver("probe").strip().splitlines()[-1])
print(json.dumps(probe, ensure_ascii=False, indent=1))
assert "ml-qwen" in probe["adapters"] and "gitleaks" in probe["adapters"] and not probe["skipped"], probe
assert "ml-v9" not in probe["adapters"], probe
assert probe["ml"]["threshold"] == THRESHOLD, probe["ml"]
assert probe["ml"]["timeout"] == 25.0 and probe["ml"]["device"].startswith("cuda"), probe["ml"]
print("사전 점검 통과. 모델 로드", probe["load_seconds"], "초")

## ⑦ 작은 실행과 시간 어림

`sessions`, `sessions_from_example`, `sessions_2`를 전체 실행과 같은 경로로 돌려 차단 건수를 본다. 이어서 8종에서 항목을 무작위로 뽑아 탐지기 전체 시간을 재고, 전체 시간을 어림한다. 작은 세 데이터는 텍스트가 짧아 그것만으로 어림하면 전체 시간이 적게 나오기 때문이다. 어림값에는 비 ML 탐지기 시간과 샤드마다 다시 읽는 모델 로드 시간이 들어 있다.

서버 CPU 참고치는 호출당 평균 4.6초, 예상 호출 수는 20,111회다.

In [ ]:
if not Path(DRIVE_DIR, "results/smoke/summary.json").is_file():
    launch("smoke")
    watch("smoke", every=60)
print(run_driver("report", "--name", "smoke"))

In [ ]:
timing_path = Path(DRIVE_DIR, "timing.json")
if not timing_path.is_file():
    assert not is_running("qwen"), "전체 실행 중에는 시간을 재지 않는다. GPU를 나눠 쓰면 타임아웃이 늘 수 있다."
    run_driver("timing", "--per-dataset", str(TIMING_SAMPLES_PER_DATASET))
timing = json.loads(timing_path.read_text(encoding="utf-8"))
print("| 데이터 | 항목 | Qwen 호출 | 표본 | 표본 중 호출 | 항목당 평균(초) | 호출당 평균(초) | 호출당 최대(초) | 표본 실패 | 어림(시간) |")
print("|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
for stem, row in timing["rows"].items():
    print(f"| {stem} | {row['items']:,} | {row['calls']:,} | {row['sampled']} | {row['sampled_calls']} | {row['item_mean_s']} | {row['call_mean_s']} | {row['call_max_s']} | {row['failures']} | {row['estimate_s'] / 3600:.2f} |")
print()
print(f"Qwen 호출 수 {timing['total_calls']:,} (예상 {EXPECTED_CALLS:,})")
print(f"호출당 시간: 평균 {timing['call_mean_s']}초, 중앙값 {timing['call_median_s']}초, 최대 {timing['call_max_s']}초 (서버 CPU 평균 4.6초). 비 ML 탐지기 시간 포함")
print(f"예상 전체 시간: 약 {timing['estimate_hours']}시간 (모델 로드 {timing['load_seconds']}초 × 샤드 약 30개 포함, 표본 어림)")
smoke = json.loads(Path(DRIVE_DIR, "results/smoke/summary.json").read_text(encoding="utf-8"))
print("작은 실행 차단:", sum(dataset["blocked"] for dataset in smoke["datasets"]), "건, failed_open:", sum(dataset["failed_open"] for dataset in smoke["datasets"]), "건")
print("어림이 터무니없거나 차단이 많으면 RUN_FULL을 켜기 전에 원인을 본다." if not RUN_FULL else "RUN_FULL이 켜져 있다.")

## ⑧ 전체 실행

서버와 같이 작업 수 1로 돌리고, 샤드는 8작업 기준으로 잘게 나눈다(세션이 끊겨도 샤드 하나만 다시 계산한다). `RUN_FULL`이 `False`면 아무것도 하지 않는다.

실행 전에 벤치마크 지문이 서버 "ML 없음" 결과와 같은지 확인한다. 샤드가 끝날 때마다 Qwen과 gitleaks가 실제로 돌았는지 검사한다. `ml-qwen`은 필수 어댑터가 아니어서, 어느 샤드에서 로드가 실패하면 Qwen 없이 점수가 나오기 때문이다. 그런 샤드가 나오면 합치지 않고 멈춘다. 다시 시작하면 그 샤드만 다시 계산한다.

In [ ]:
if not RUN_FULL:
    print("RUN_FULL = False: 전체 실행을 시작하지 않았다. ⑦을 본 뒤 ①에서 True로 바꾸고 ①과 ⑧을 실행한다.")
elif Path(DRIVE_DIR, "results/qwen/summary.json").is_file():
    print("전체 실행 결과가 이미 있다. ⑩을 실행한다.")
else:
    launch("qwen")

## ⑨ 진행 확인

5분마다 진행과 차단 건수를 보여 주고, 실행이 끝나면 멈춘다. 이 셀을 중단하거나 브라우저를 닫아도 평가는 계속되지만, Colab이 런타임을 회수하면 멈춘다. 그때는 ①부터 다시 실행하고 ⑧로 이어 간다.

In [ ]:
if not Path(DRIVE_DIR, "status_qwen.json").is_file() and not is_running("qwen"):
    print("전체 실행을 시작한 적이 없다. ⑧을 먼저 실행한다.")
else:
    watch("qwen")

## ⑩ 결과

데이터별 `blocked`, `failed_open`, 어댑터 실패 수를 보여 주고 실행 조건을 `RUN_NOTES.md`로 함께 저장한다. 비밀값은 출력하지 않는다.

- **blocked**: 탐지기 오류나 타임아웃으로 Final Guard가 막았다고 본 항목 수다. 그 항목의 정답은 `over`로 채점된다.
- **adapter 실패**: 샤드 로그의 `어댑터:예외 형식`별 횟수다. 타임아웃은 `ml-qwen:RuntimeError`로 나오며 다른 Qwen 실패와 구분되지 않는다.

묶음별 표는 서버에서 만든다. Drive의 `qwen-v3-cca44c4/results/qwen/`에서 `summary.json`, `meta.json`, `RUN_NOTES.md`를 받아 서버 `runs/e2e-20261106/results/qwen/`에 두고 `report.py base v10 qwen`을 실행한다. `examples/`는 비밀값이 있으니 필요할 때만 옮긴다.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(run_driver("report", "--name", "qwen")))
display(Markdown(Path(DRIVE_DIR, "results/qwen/RUN_NOTES.md").read_text(encoding="utf-8")))